In [ ]:
import xarray as xr

file_path = r"Your_Path.nc"

ds = xr.open_dataset(file_path)
print(ds)

In [ ]:
ds["t2m_c"] = ds["t2m"] - 273.15

temp_series = ds["t2m_c"].mean(dim=["latitude", "longitude"])

In [ ]:
df = temp_series.to_dataframe().reset_index()

df.head()

In [ ]:
for i in range(1,25):
    df[f"temp_t-{i}"] = df["t2m_c"].shift(i)

df = df.dropna()

In [ ]:
df["target"] = df["t2m_c"].shift(-24)

df = df.dropna()

In [ ]:
split = int(len(df)*0.8)

train = df[:split]
test = df[split:]

X_train = train[[f"temp_t-{i}" for i in range(1,25)]]
y_train = train["target"]

X_test = test[[f"temp_t-{i}" for i in range(1,25)]]
y_test = test["target"]

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200)
rf.fit(X_train, y_train)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor()
gbr.fit(X_train, y_train)

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor()
xgb.fit(X_train, y_train)

In [ ]:
from sklearn.svm import SVR

svr = SVR()
svr.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate(name, y_true, pred):
    mae = mean_absolute_error(y_true, pred)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    print(name, "MAE:", mae)
    print(name, "RMSE:", rmse)
    print("-----------")

In [ ]:
evaluate("Linear Regression", y_test, lr.predict(X_test))
evaluate("Random Forest", y_test, rf.predict(X_test))
evaluate("Gradient Boosting", y_test, gbr.predict(X_test))
evaluate("XGBoost", y_test, xgb.predict(X_test))
evaluate("SVR", y_test, svr.predict(X_test))

In [ ]:
## retest on 1 week data to check how good model got trained


In [ ]:
import xarray as xr

test_file = r"file_path"

ds_test = xr.open_dataset(test_file)
print(ds_test)

In [ ]:
ds_test["t2m_c"] = ds_test["t2m"] - 273.15
temp_test = ds_test["t2m_c"].mean(dim=["latitude","longitude"])

In [ ]:
df_test = temp_test.to_dataframe().reset_index()

In [ ]:
for i in range(1,25):
    df_test[f"temp_t-{i}"] = df_test["t2m_c"].shift(i)

df_test = df_test.dropna()

In [ ]:
X_test_new = df_test[[f"temp_t-{i}" for i in range(1,25)]]
y_test_new = df_test["t2m_c"]

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate(name, y_true, pred):
    mae = mean_absolute_error(y_true, pred)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    print(name, "MAE:", mae)
    print(name, "RMSE:", rmse)
    print("-----------")

In [ ]:
evaluate("Linear Regression", y_test_new, lr.predict(X_test_new))
evaluate("Random Forest", y_test_new, rf.predict(X_test_new))
evaluate("Gradient Boosting", y_test_new, gbr.predict(X_test_new))
evaluate("XGBoost", y_test_new, xgb.predict(X_test_new))
evaluate("SVR", y_test_new, svr.predict(X_test_new))

In [ ]:
#plottings

import matplotlib.pyplot as plt

def plot_all_models(y_true, preds_dict, title):
    plt.figure(figsize=(14,6))
    
    plt.plot(y_true.values, label="Actual", linewidth=2, color="black")
    
    for name, pred in preds_dict.items():
        plt.plot(pred, label=name, alpha=0.8)
    
    plt.title(title)
    plt.xlabel("Time Steps")
    plt.ylabel("Temperature (°C)")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
## 1 — 1-Month Benchmark Plot

preds_case1 = {
    "Linear Regression": pred_lr,
    "Random Forest": pred_rf,
    "Gradient Boosting": pred_gbr,
    "XGBoost": pred_xgb,
    "SVR": pred_svr
}

plot_all_models(y_test, preds_case1, "1-Month Training Benchmark")

In [ ]:
##Case 2 — 6-Month Training Plot
preds_case2 = {
    "Linear Regression": lr.predict(X_test),
    "Random Forest": rf.predict(X_test),
    "Gradient Boosting": gbr.predict(X_test),
    "XGBoost": xgb.predict(X_test),
    "SVR": svr.predict(X_test)
}

plot_all_models(y_test, preds_case2, "6-Month Training Benchmark")

In [ ]:
##Case 3 — Jan 2026 Generalization Plot
preds_case3 = {
    "Linear Regression": pred_lr_jan,
    "Random Forest": pred_rf_jan,
    "Gradient Boosting": pred_gbr_jan,
    "XGBoost": pred_xgb_jan,
    "SVR": pred_svr_jan
}

plot_all_models(y_test_new, preds_case3, "Jan 2026 Generalization Test")